# Cricbuzz LiveStats: Data Fetching and Exploration





#### **Project Type**    - API Intigration and Data Fetching
##### **Contribution**    - Individual
##### **Team Member 1 -** Mriganka Das
##### **Team Member 2 -**
##### **Team Member 3 -**
##### **Team Member 4 -**

## Summary

This notebook documents how the project's data gets from the **Cricbuzz API** into the local **SQLite** database, then explores it.

**Pipeline:** API → raw JSON cache (`data/api_cache.db`) → parsers → SQL tables (`data/cricket.db`) → analysis

**Quota-safe by default.** With `USE_API = False` (Section 2) every cell reads from the saved cache and the database, so it runs with **zero API calls**. Set it to `True` only when we want one fresh call.

## 1. Imports and environment setup

In [2]:
# Standard library imports
import os
import sys
import json
import sqlite3
import subprocess
from pathlib import Path

# Third-party imports
import pandas as pd
from dotenv import load_dotenv

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)

Python version: 3.10.2
Pandas version: 2.3.3


In [3]:
# Find the project root from this notebook's location
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "notebooks").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Make the project's own packages importable (services, api, database, ...)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables from the project root
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

import config  # the same settings the Streamlit app uses

DB_PATH = PROJECT_ROOT / "data" / "cricket.db"

print("Project root:", PROJECT_ROOT)
print("Environment file exists:", ENV_PATH.is_file())
print("Database file exists:", DB_PATH.is_file())

# Never print secret values; only whether they are set
print("API key configured:", bool(config.RAPIDAPI_KEY))
print("API host:", config.RAPIDAPI_HOST)

Project root: c:\Users\MRIGANKA DAS\OneDrive\Desktop\cricbuzz_livestats
Environment file exists: True
Database file exists: True
API key configured: True
API host: cricbuzz-cricket2.p.rapidapi.com


## 2. Settings

In [4]:
# False = read only from the saved cache (no API quota used).
# True  = make ONE fresh call for the recent-matches list, then save it.
USE_API = False

## 3. Fetch the match list

The API returns nested JSON: `typeMatches → seriesMatches → seriesAdWrapper → matches → matchInfo`. `parse_matches` flattens it into simple rows. Every API response is saved in the raw cache first, so it is never paid for twice.

In [5]:
from api.cricbuzz_client import CricbuzzClient, CricbuzzAPIError
from services import raw_store
from services.match_service import parse_matches

LIST_PATH = "/matches/v1/recent"

payload = raw_store.get(LIST_PATH)          # newest saved copy, any age
if USE_API:
    try:
        payload = CricbuzzClient().recent_matches()
        raw_store.put(LIST_PATH, None, payload)
        print("Fetched fresh data from the API and saved it to the cache.")
    except CricbuzzAPIError as exc:
        print("API call failed, using the cache instead:", exc)

if payload is None:
    print("No cached match list yet. Run:  python scripts/harvest.py --max-calls 10")
else:
    print("Match list ready. Top-level keys:", list(payload))

Match list ready. Top-level keys: ['typeMatches', 'filters', 'appIndex', 'responseLastUpdated']


In [6]:
matches_df = pd.DataFrame(parse_matches(payload)) if payload else pd.DataFrame()
print(f"{len(matches_df)} matches in the cached list")
matches_df.head(10)

73 matches in the cached list


,match_id,series,description,format,state,status,team1,team2,score1,score2,venue
0,170723,"Pakistan ODI Tri-Series, 2026",2nd Match,ODI,Complete,Pakistan won by 87 runs,Pakistan,England,302/9 (49.6 ov),215/10 (38.4 ov),"Rawalpindi Cricket Stadium, Rawalpindi"
1,170720,"Pakistan ODI Tri-Series, 2026",1st Match,ODI,Complete,Pakistan won by 37 runs,Pakistan,Sri Lanka,305/10 (49.5 ov),268/10 (48.1 ov),"Rawalpindi Cricket Stadium, Rawalpindi"
2,172681,"ODI Tri-Nation Series in UAE, 2026",3rd Match,ODI,Complete,Afghanistan won by 7 wkts,Bangladesh,Afghanistan,160/10 (41.2 ov),161/3 (35.3 ov),"Sharjah Cricket Stadium, Sharjah"
3,172680,"ODI Tri-Nation Series in UAE, 2026",2nd Match,ODI,Complete,Afghanistan won by 158 runs,Afghanistan,Zimbabwe,294/10 (49.6 ov),136/10 (38.4 ov),"Sheikh Zayed Stadium, Abu Dhabi"
4,172677,"ODI Tri-Nation Series in UAE, 2026",1st Match,ODI,Complete,Zimbabwe won by 99 runs,Zimbabwe,Bangladesh,271/8 (49.6 ov),172/10 (33.4 ov),"Sheikh Zayed Stadium, Abu Dhabi"
5,173973,ICC Men's T20 World Cup Africa Sub Regional Qualifier C 2026,6th Match,T20,Complete,Nigeria won by 9 wkts,Zambia,Nigeria,142/9 (19.6 ov),146/1 (14.5 ov),"Nigeria Cricket Federation Oval 1, Abuja"
6,173967,ICC Men's T20 World Cup Africa Sub Regional Qualifier C 2026,4th Match,T20,Complete,Mozambique won by 28 runs,Mozambique,Gambia,148/6 (19.6 ov),120/8 (19.6 ov),"Nigeria Cricket Federation Oval 1, Abuja"
7,173971,ICC Men's T20 World Cup Africa Sub Regional Qualifier C 2026,5th Match,T20,Complete,Uganda won by 5 wkts,Lesotho,Uganda,87/10 (19.3 ov),89/5 (12.3 ov),"Nigeria Cricket Federation Oval 2, Abuja"
8,173950,ICC Men's T20 World Cup Africa Sub Regional Qualifier C 2026,2nd Match,T20,Complete,Lesotho won by 74 runs,Lesotho,Mozambique,169/8 (19.6 ov),95/10 (18.2 ov),"Nigeria Cricket Federation Oval 1, Abuja"
9,173957,ICC Men's T20 World Cup Africa Sub Regional Qualifier C 2026,3rd Match,T20,Complete,Nigeria won by 160 runs,Nigeria,Gambia,262/1 (19.6 ov),102/10 (17.2 ov),"Nigeria Cricket Federation Oval 2, Abuja"


In [7]:
if not matches_df.empty:
    display(matches_df.groupby(["format", "state"]).size().rename("matches").reset_index())

,format,state,matches
0,ODI,Complete,10
1,T20,Complete,43
2,TEST,Complete,20


## 4. Look at the raw scorecard JSON

Before writing a parser it is worth looking at the real structure. This API uses **lowercase** field names (`batsman`, `bowler`, `outdec`, `strkrate`), unlike the camelCase match list.

In [8]:
sample = None
if not matches_df.empty:
    for match_id in matches_df["match_id"]:
        sample = raw_store.get(f"/mcenter/v1/{match_id}/scard")
        if sample:
            print("Using cached scorecard for match", match_id)
            break

if sample is None:
    print("No cached scorecards yet. Harvest some first.")
else:
    print("Top-level keys:", list(sample))
    first_innings = sample["scorecard"][0]
    print("Innings keys  :", list(first_innings))
    print("First batter  :")
    print(json.dumps(first_innings["batsman"][0], indent=2))

Using cached scorecard for match 170723
Top-level keys: ['scorecard', 'ismatchcomplete', 'appindex', 'status', 'responselastupdated']
Innings keys  : ['inningsid', 'batsman', 'bowler', 'fow', 'extras', 'pp', 'score', 'wickets', 'overs', 'runrate', 'batteamname', 'batteamsname', 'isdeclared', 'isfollowon', 'ballnbr', 'rpb', 'partnership']
First batter  :
{
  "id": 10863,
  "balls": 1,
  "runs": 0,
  "fours": 0,
  "sixes": 0,
  "strkrate": "0",
  "name": "Fakhar Zaman",
  "nickname": "Fakhar Zaman",
  "iscaptain": false,
  "iskeeper": false,
  "outdec": "c Joe Root b Gus Atkinson",
  "videotype": "",
  "videourl": "",
  "videoid": 0,
  "planid": 0,
  "imageid": 0,
  "premiumvideourl": "",
  "iscbplusfree": false,
  "ispremiumfree": false,
  "inmatchchange": "",
  "isoverseas": false,
  "playingxichange": ""
}


## 5. Build the database from the cache

`scripts/build_from_cache.py` reads the cached JSON, parses it, and fills the SQL tables (matches, innings, batting and bowling scorecards, results, toss and venue details). It uses **no API calls** and is safe to re-run, because inserts are upserts.

In [9]:
result = subprocess.run(
    [sys.executable, "scripts/build_from_cache.py"],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("BUILD FAILED")
    print(result.stderr)

Matches saved: 73
Innings saved: 49 | match-info files applied: 24
Match results understood from status text: 119
  matches with a winner: 104
  matches with toss info: 26
  venues with a country: 14
  venues with capacity: 10
  innings: 53
  batting rows: 449



## 6. Database overview

In [10]:
def query(sql, params=()):
    """Run a read-only SQL query and return a DataFrame."""
    uri = DB_PATH.as_uri() + "?mode=ro"
    conn = sqlite3.connect(uri, uri=True)
    try:
        return pd.read_sql_query(sql, conn, params=params)
    finally:
        conn.close()


TABLES = ["teams", "players", "venues", "series", "matches", "innings",
          "batting_scorecard", "bowling_scorecard", "fielding_stats",
          "player_career_stats"]
counts = pd.DataFrame(
    {"table": TABLES,
     "rows": [query(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in TABLES]})
counts

,table,rows
0,teams,116
1,players,355
2,venues,71
3,series,4
4,matches,121
5,innings,53
6,batting_scorecard,449
7,bowling_scorecard,306
8,fielding_stats,0
9,player_career_stats,7


In [11]:
query("""
    SELECT m.match_date, m.match_format, t1.team_name AS team1, t2.team_name AS team2,
           w.team_name AS winner, m.victory_margin, m.victory_type, m.status
    FROM matches m
    JOIN teams t1 ON t1.team_id = m.team1_id
    JOIN teams t2 ON t2.team_id = m.team2_id
    LEFT JOIN teams w ON w.team_id = m.winner_id
    WHERE m.cb_match_id IS NOT NULL
    ORDER BY m.match_date DESC
    LIMIT 15
""")

,match_date,match_format,team1,team2,winner,victory_margin,victory_type,status
0,2026-10-21,T20I,Sydney Sixers Women,Perth Scorchers Women,Sydney Sixers Women,15,runs,Sydney Sixers Women won by 15 runs
1,2026-10-21,ODI,Kenya,Denmark,Kenya,103,runs,Kenya won by 103 runs
2,2026-10-21,T20I,Northern Cape,South Western Districts,South Western Districts,6,wickets,South Western Districts won by 6 wkts
3,2026-10-21,T20I,Limpopo,North West - Eastvaal Renault Dragons,North West - Eastvaal Renault Dragons,5,wickets,North West - Eastvaal Renault Dragons won by 5 wkts
4,2026-10-21,T20I,Lesotho,Uganda,Uganda,5,wickets,Uganda won by 5 wkts
5,2026-10-21,T20I,Mozambique,Gambia,Mozambique,28,runs,Mozambique won by 28 runs
6,2026-10-21,T20I,Zambia,Nigeria,Nigeria,9,wickets,Nigeria won by 9 wkts
7,2026-10-21,ODI,Bangladesh,Afghanistan,Afghanistan,7,wickets,Afghanistan won by 7 wkts
8,2026-10-20,T20I,Australia Women,New Zealand Women,Australia Women,40,runs,Australia Women won by 40 runs
9,2026-10-20,T20I,India Women,Zimbabwe Women,India Women,92,runs,India Women won by 92 runs


## 7. Data quality checks

Good analysis starts with knowing what is missing. Each row below is one check.

In [12]:
checks = {
    "Sample/seed players still in database":
        "SELECT COUNT(*) AS n FROM players WHERE full_name LIKE 'Sample Player%'",
    "Real matches (from the API)":
        "SELECT COUNT(*) AS n FROM matches WHERE cb_match_id IS NOT NULL",
    "Real matches with a winner":
        "SELECT COUNT(*) AS n FROM matches WHERE cb_match_id IS NOT NULL AND winner_id IS NOT NULL",
    "Real matches with toss info":
        "SELECT COUNT(*) AS n FROM matches WHERE cb_match_id IS NOT NULL AND toss_winner_id IS NOT NULL",
    "Players with no playing role recorded":
        "SELECT COUNT(*) AS n FROM players WHERE playing_role IS NULL",
    "Innings with no batting rows":
        "SELECT COUNT(*) AS n FROM innings WHERE innings_id NOT IN (SELECT innings_id FROM batting_scorecard)",
    "Duplicate Cricbuzz match IDs (should be 0)":
        "SELECT COUNT(*) AS n FROM (SELECT cb_match_id FROM matches WHERE cb_match_id IS NOT NULL GROUP BY cb_match_id HAVING COUNT(*) > 1)",
}
pd.DataFrame({"check": list(checks),
              "result": [query(sql)["n"][0] for sql in checks.values()]})

,check,result
0,Sample/seed players still in database,6
1,Real matches (from the API),119
2,Real matches with a winner,102
3,Real matches with toss info,24
4,Players with no playing role recorded,349
5,Innings with no batting rows,0
6,Duplicate Cricbuzz match IDs (should be 0),0


In [13]:
# Foreign-key integrity: an empty table means every reference is valid
fk = query("PRAGMA foreign_key_check")
print("Foreign-key violations:", len(fk))
fk

Foreign-key violations: 0


,table,rowid,parent,fkid


## 8. Exploratory Data Analysis and Visualization

Charts come from the same `chart_service` the Streamlit app uses, so the notebook and the app always agree. Seed rows are excluded. If a figure does not display in VS Code, run `pip install nbformat` once.

In [14]:
from services import chart_data_service as data
from services import chart_service as charts

top = data.top_scorers("All", 10)
display(top)
if not top.empty:
    charts.ranking_bar(top, "player", "runs", "Top run scorers (harvested matches)").show()

,player,runs
0,Matt Renshaw,272
1,Isaac Danladi,176
2,Selim Salau,155
3,Chadwick Walton,145
4,Hammad Mirza,136
5,Hasan Nawaz,132
6,Moeen Ali,130
7,Ibrahim Zadran,124
8,Salman Agha,123
9,Aiden Markram,112


### Insight: Top Run Scorers (Harvested Matches)
#### Chart overview
This horizontal bar chart compares the total runs recorded for the top 10 players in the harvested cricket-match dataset.
#### Key observations
- Matt Renshaw ranks first with 272 runs.
- Isaac Danladi ranks second with 176 runs.
- Selim Salau ranks third with 155 runs.
- Chadwick Walton follows with 145 runs.
- Hammad Mirza has 136 runs, followed by Hasan Nawaz with 132 runs.
- The remaining players shown are Moeen Ali (130), Ibrahim Zadran (124), Salman Agha (123), and Aiden Markram (112).
#### Main insight
Matt Renshaw leads the chart by 96 runs over Isaac Danladi (272 − 176), showing a clear gap between the top two players in this dataset. Isaac Danlani is 21 runs ahead of Selim Salau (176 − 155). The displayed values decrease from 272 to 112 runs across the top 10.
#### Interpretation
The chart identifies the leading run scorers among the matches currently stored in the project. It helps compare player contributions and highlights which players accumulated the most runs in the available data.

In [15]:
wins = data.wins_by_team()
display(wins.head(10))
if not wins.empty:
    charts.build_chart("Pie", wins, "team", "wins", "Wins by team").show()

,team,wins
0,Sweden Women,4
1,Adelaide Strikers Women,3
2,India Women,3
3,Australia Women,3
4,West Indies Champions,3
5,Uganda,2
6,Nigeria,2
7,Austria Women,2
8,Bulgaria Women,2
9,Melbourne Renegades Women,2


### Insight: Wins by Team
#### Chart overview
The donut chart compares the recorded win counts for teams represented in the harvested cricket dataset. Each slice shows a team's share of the wins included in this chart.
#### Key observations
- Sweden Women has the largest displayed share at 7.41%.
- Adelaide Strikers Women is shown with 5.56%.
- Several teams have a displayed share of 5.56%, including India Women and Australia Women.
- Many other teams have slices labelled 3.70%, indicating smaller and similar shares in the chart.
- The legend includes a broad mix of women's teams, domestic/franchise teams, and regional teams.
#### Main insight
Sweden Women has the largest individual slice in the displayed chart, at 7.41%. However, the chart is spread across many teams, and most visible slices are relatively small (around 3.70%–5.56%). This suggests that the wins represented in the harvested dataset are distributed among numerous teams rather than dominated by one team.
#### Interpretation
This visualization provides a quick overview of which teams account for the largest share of wins in the current dataset. It is useful for identifying the most frequently winning teams among the matches that have been collected.

In [16]:
by_format = query("""
    SELECT m.match_format AS format,
           COUNT(DISTINCT m.match_id) AS matches,
           ROUND(AVG(i.total_runs), 1) AS avg_innings_total
    FROM matches m
    LEFT JOIN innings i ON i.match_id = m.match_id
    WHERE m.cb_match_id IS NOT NULL
    GROUP BY m.match_format
""")
by_format

,format,matches,avg_innings_total
0,ODI,14,236.0
1,T20I,64,150.8
2,Test,41,298.5


In [17]:
scatter = data.average_vs_strike_rate(min_balls=30)
display(scatter.head(10))
if len(scatter) >= 2:
    charts.avg_vs_strike_scatter(scatter, "Batting average vs strike rate").show()
else:
    print("Not enough batters yet. Harvest more matches and re-run Section 5.")

,player,runs,strike_rate,average
0,Sanju Samson,72,240.0,36.0
1,Tilak Varma,50,161.3,25.0
3,Nic Maddinson,48,67.6,24.0
4,Travis Head,20,54.1,10.0
5,Marnus Labuschagne,74,63.8,37.0
6,Steven Smith,52,55.3,26.0
7,Matt Renshaw,272,54.8,136.0
8,Alex Carey,38,45.8,19.0
9,Beau Webster,43,95.6,43.0
10,Pat Cummins,75,68.2,75.0


### Insight: Batting Average vs Strike Rate
#### Chart overview
This scatter plot compares players' batting averages (horizontal axis) with their strike rates (vertical axis). Each point represents a player record; the point size may encode an additional measure, depending on the chart configuration.
#### Key observations
- Most points are clustered at batting averages of roughly 20–70 and strike rates of about 50–120.
- A smaller number of points have strike rates above 150, including one around 240.
- Several points appear at higher batting averages (approximately 100–180), with different strike rates.
- There are visible outliers, including a point near an average of 136 and strike rate of 55, and another near an average of 176 and strike rate of 184.
- The points are widely dispersed rather than following a clear straight-line pattern.
#### Main insight
The plot suggests that batting average and strike rate do not have a simple, consistent linear relationship in this dataset. Many players sit in a moderate range for both measures, while some combine a high average with a lower strike rate or a high strike rate with a more modest average. These profiles reflect different scoring patterns.
#### Interpretation
- Batting average indicates runs scored per dismissal under the dataset's calculation.
- Strike rate indicates the rate of scoring, typically runs per 100 balls for batting.
- A player with a higher average is not automatically the fastest scorer, and a higher strike rate does not automatically mean a higher average. Both measures should be considered together.

## 9. Conclusion

This notebook showed the full data journey behind **Cricbuzz LiveStats**: the Cricbuzz RapidAPI feeds a local raw cache, parsers turn those responses into clean records, and the records are stored in a SQLite database that the Streamlit app queries.

**What we built and verified**
- **A quota-safe pipeline** (API → raw cache → parsers → SQLite). Every API response is cached, so the app and this notebook run without spending any API quota. This matters because the free tier has a daily limit.
- **A loaded, queryable database.** It holds about 119 real matches, around 450 players with career statistics, and batting and bowling scorecards, all stored with foreign keys and parameterised SQL.
- **Analytics and visualisation.** All 25 SQL questions run on this data and the charts are built from the same database. Matches, scorecards, top players, venues and trends can all be explored from the app.
- **Reliability.** A pytest suite covers the parsers, ingestion and history-based queries, and the app logs errors and shows friendly messages instead of crashing.

**Limitations (stated honestly)**
- The harvested data covers only about two weeks of matches, because the free API tier limits how much history can be collected. Questions that need long history, such as the quarterly trend and career phase (Q25), use shorter weekly periods and relaxed minimums. Their original logic is verified on synthetic multi-year data in the tests.
- Some details are still thin because the API does not provide them in scorecards: player roles and batting/bowling styles for many players, and fielding statistics (catches and stumpings).

